<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex11.1-camera-navigation/Ex11.1_10_camera_navigation.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_11.1 — Camera Navigation with the Waveshare JetRacer

**Applied PINN for Energy · Aalborg University**

Run this **on the car**, in JupyterLab at `http://<ip>:8888`.

**The car:** Waveshare JetRacer Pro — Jetson Nano 4 GB, IMX219-160 camera
(8 MP, 160° FOV), Ackermann steering on a 6 kg·cm servo, 4WD through front and
rear differentials on RC380 brushed motor drive, adjustable oil-filled
shocks with independent suspension, and a 8.4 V pack of four 18650s in 2S2P.
There is no motor encoder.

Before you open this notebook you must have completed `Ex_11.1_SETUP.md`:
the Waveshare image is flashed, the Nano is in **5 W mode**, and
`/jetracer/notebooks/basic_motion.ipynb` turns the wheels. If the car has never
driven, stop here — debugging a network on a car that was never driving is a
wasted afternoon.

---

### What you are being scored on

| | |
|---|---|
| **Navigation Index** | `N = T0 / (T_median + 2c)` — higher is better, `N > 1` beats your car's baseline |
| **Blind distance** | `d = v / f` — must be reported; exceeding half the narrowest clearance caps you at `N = 1.00` |
| **Gates** | 3 clean laps · loop rate measured · failsafe demonstrated · held-out validation |

Your car has a number. Your dataset, your model and your results are all tagged
with it, and the leaderboard is normalised per car so hardware luck does not
decide the trophy.

## 0 · Team and car identity

Everything this notebook writes is tagged with these two fields. Set them once.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex11.1-camera-navigation/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
TEAM = "team-01"          # <- your team id
CAR  = 1                  # <- the number painted on your car AND on your SD card
COURSE_LENGTH_M = 20.0    # <- measured length of one lap
NARROWEST_CLEARANCE_M = 0.35   # <- narrowest gap on the course, measured

import os, json, time, datetime, pathlib

RUN_ID  = f"{TEAM}_car{CAR:02d}"
DATA    = pathlib.Path(f"/home/jetson/ex11/{RUN_ID}")
(DATA / "dataset").mkdir(parents=True, exist_ok=True)
(DATA / "logs").mkdir(parents=True, exist_ok=True)

META = {"team": TEAM, "car": CAR, "course_length_m": COURSE_LENGTH_M,
        "narrowest_clearance_m": NARROWEST_CLEARANCE_M,
        "created": datetime.datetime.now().isoformat(timespec="seconds")}
(DATA / "meta.json").write_text(json.dumps(META, indent=2))
print(RUN_ID, "->", DATA)

## 1 · Hardware check

Confirms the four things that break silently. Run it after **every** reboot —
the 5 W power mode does not always survive one.

In [ ]:
import subprocess

def check(label, fn):
    try:
        ok, detail = fn()
    except Exception as e:
        ok, detail = False, f"{type(e).__name__}: {e}"
    print(f"[{'OK ' if ok else 'FAIL'}] {label:28s} {detail}")
    return ok

def _power_mode():
    out = subprocess.check_output(["nvpmodel", "-q"]).decode()
    is5w = "MAXN" not in out.upper()
    return is5w, out.strip().replace("\n", " | ")

def _jetracer():
    from jetracer.nvidia_racecar import NvidiaRacecar
    car = NvidiaRacecar()
    return True, f"{type(car).__module__}"

def _camera():
    from jetcam.csi_camera import CSICamera
    cam = CSICamera(width=224, height=224, capture_fps=30)
    img = cam.read()
    cam.running = False
    return img is not None, f"frame {getattr(img, 'shape', None)}"

def _torch():
    import torch
    return torch.cuda.is_available(), f"torch {torch.__version__}, cuda={torch.cuda.is_available()}"

results = [
    check("5 W power mode",   _power_mode),
    check("jetracer package", _jetracer),
    check("CSI camera",       _camera),
    check("torch + CUDA",     _torch),
]
print()
print("ALL CHECKS PASSED" if all(results) else "*** FIX THE FAILURES ABOVE BEFORE CONTINUING ***")

> **Trap 1 — Waveshare, not NVIDIA.** If `jetracer` imports cleanly and the
> wheels still do not turn, you have NVIDIA's motor code. Remove the `jetracer`
> folder and reinstall the Waveshare version.
>
> **Trap 2 — 5 W mode.** If the check above fails, set it *now*. Skipping it
> means the car resets under load, and it will happen during your scored run.

## 2 · Data collection

The stock pipeline learns a regression from image to a single target point:
you click where you want the car to aim, and the click becomes the label.

**Collect deliberately.** A dataset of 200 frames from the racing line teaches
the car to follow the racing line and nothing else. It has never seen the
recovery it needs when it drifts wide. Deliberately drive off-line and label
the correction — that is where the useful gradient lives.

In [ ]:
import ipywidgets, traitlets, uuid
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True

image_widget  = ipywidgets.Image(format="jpeg")
count_widget  = ipywidgets.IntText(description="frames", value=0)
traitlets.dlink((camera, "value"), (image_widget, "value"), transform=bgr8_to_jpeg)

def _save(_, content, msg):
    """Click the target point in the image; x,y are baked into the filename."""
    if content["event"] != "click":
        return
    x, y = content["eventData"]["offsetX"], content["eventData"]["offsetY"]
    name = f"xy_{x:03d}_{y:03d}_{uuid.uuid1()}.jpg"
    (DATA / "dataset" / name).write_bytes(image_widget.value)
    count_widget.value = len(list((DATA / "dataset").glob("*.jpg")))

try:
    from jupyter_clickable_image_widget import ClickableImageWidget
    click_widget = ClickableImageWidget(width=224, height=224)
    traitlets.dlink((camera, "value"), (click_widget, "value"), transform=bgr8_to_jpeg)
    click_widget.on_msg(_save)
    display(ipywidgets.VBox([click_widget, count_widget]))
except ImportError:
    print("jupyter_clickable_image_widget not installed — use the stock "
          "Waveshare data_collection.ipynb and point DATA/dataset at its output.")
    display(ipywidgets.VBox([image_widget, count_widget]))

**Collection checklist** — tick all five before you move on.

1. At least 300 frames.
2. Both directions around the course.
3. Off-line recovery frames, not only the racing line.
4. Every corner represented, not just the easy ones.
5. Lighting as it will be at competition time. Do not collect at 09:00 and race at 15:00.

## 3 · Train, and hold data out

The held-out split is a competition gate, not a suggestion. A model evaluated
only on its training frames tells you how well it memorised the afternoon.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, torchvision, glob, random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class TargetPointDataset(Dataset):
    """Filename encodes the clicked target: xy_XXX_YYY_<uuid>.jpg"""
    NORM_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    NORM_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __init__(self, paths, augment=False):
        self.paths, self.augment = paths, augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        p = self.paths[i]
        stem = os.path.basename(p).split("_")
        x, y = float(stem[1]), float(stem[2])
        img = Image.open(p).convert("RGB")
        t = torchvision.transforms.functional.to_tensor(img)
        if self.augment and random.random() < 0.5:      # horizontal flip
            t = torch.flip(t, dims=[2]); x = 223.0 - x
        t = (t - self.NORM_MEAN) / self.NORM_STD
        # map pixel coords to [-1, 1]
        return t, torch.tensor([x / 112.0 - 1.0, y / 112.0 - 1.0], dtype=torch.float32)

paths = sorted(glob.glob(str(DATA / "dataset" / "*.jpg")))
random.Random(0).shuffle(paths)
split = int(0.8 * len(paths))
train_paths, val_paths = paths[:split], paths[split:]
print(f"{len(train_paths)} train / {len(val_paths)} held out")
assert len(val_paths) >= 40, "collect more data — the held-out split is too small to mean anything"

train_dl = DataLoader(TargetPointDataset(train_paths, augment=True), batch_size=16, shuffle=True)
val_dl   = DataLoader(TargetPointDataset(val_paths), batch_size=16)

In [ ]:
device = torch.device("cuda")
model = torchvision.models.resnet18(pretrained=True)
model.fc = nn.Linear(512, 2)
model = model.to(device)

opt = optim.Adam(model.parameters(), lr=1e-4)
lossf = nn.MSELoss()
EPOCHS = 30

best = float("inf")
history = []
for ep in range(EPOCHS):
    model.train(); tr = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); l = lossf(model(xb), yb); l.backward(); opt.step()
        tr += float(l) * len(xb)
    model.eval(); va = 0.0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            va += float(lossf(model(xb), yb)) * len(xb)
    tr /= len(train_dl.dataset); va /= len(val_dl.dataset)
    history.append({"epoch": ep, "train_mse": tr, "val_mse": va})
    if va < best:
        best = va
        torch.save(model.state_dict(), DATA / "model_best.pth")
    print(f"epoch {ep:02d}  train {tr:.5f}   val {va:.5f}{'   <- saved' if va == best else ''}")

json.dump({"history": history, "best_val_mse": best},
          open(DATA / "logs" / "training.json", "w"), indent=2)
print(f"\nHELD-OUT MSE = {best:.5f}   (report this number)")

## 4 · The control loop, instrumented

This replaces the stock `road_following.ipynb`. Three things are added, and all
three are competition gates:

- **A measured loop rate.** Not assumed, not the camera's nominal FPS — measured.
- **A failsafe.** A frame older than `STALE_S` stops the car. No exceptions.
- **A log.** Buffered in memory, written once at the end. An `fsync` inside the
  loop costs you frames.

In [ ]:
from collections import deque
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

STEER_GAIN    = 0.85     # tune: how hard the car turns for a given target offset
STEER_BIAS    = 0.00     # tune: mechanical trim; a car that drifts on zero is not trimmed
THROTTLE      = 0.35     # keep <= 0.4 until the loop rate is measured
STALE_S       = 0.20     # a frame older than this means STOP
LOOP_BUDGET   = deque(maxlen=200)

model.eval()
MEAN = TargetPointDataset.NORM_MEAN.to(device)
STD  = TargetPointDataset.NORM_STD.to(device)

def preprocess(bgr):
    import cv2
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).to(device)
    return ((t - MEAN) / STD).unsqueeze(0)

def stop():
    car.throttle = 0.0
    car.steering = 0.0

log = []
running = False

def drive_loop(duration_s=60.0):
    """Run the policy for duration_s, logging every iteration."""
    global running
    running = True
    stop()
    t_start = time.monotonic()
    last_frame_t = t_start
    try:
        while running and (time.monotonic() - t_start) < duration_s:
            t0 = time.monotonic()
            frame = camera.value
            if frame is None or (t0 - last_frame_t) > STALE_S:
                stop()                                   # FAILSAFE
                log.append({"t": t0 - t_start, "stale": 1})
                continue
            last_frame_t = t0
            with torch.no_grad():
                x, y = model(preprocess(frame))[0].tolist()
            steer = max(-1.0, min(1.0, x * STEER_GAIN + STEER_BIAS))
            car.steering = steer
            car.throttle = THROTTLE
            dt = time.monotonic() - t0
            LOOP_BUDGET.append(dt)
            log.append({"t": t0 - t_start, "dt": dt, "x": x, "y": y,
                        "steer": steer, "throttle": THROTTLE, "stale": 0})
    finally:
        stop()
        running = False
    return log

print("ready — one person drives, another watches with a hand on the stop")

### Measure the loop rate before you race

`f` is the median of the achieved loop period, not the best case. The blind
distance follows: at 2 m/s and 20 Hz the car travels 10 cm between decisions,
and if that exceeds half your narrowest clearance the gate caps you.

In [ ]:
_ = drive_loop(duration_s=15.0)       # a short shakedown, not a scored run

import statistics
dts = [e["dt"] for e in log if "dt" in e]
f_med  = 1.0 / statistics.median(dts)
f_p05  = 1.0 / (sorted(dts)[int(0.95 * len(dts))])     # pessimistic: 95th pct period
v_est  = 1.8                                            # measure this on the course
d_blind = v_est / f_med

print(f"loop rate   median {f_med:5.1f} Hz    worst-5% {f_p05:5.1f} Hz")
print(f"blind dist  {d_blind*100:5.1f} cm at {v_est} m/s")
print(f"clearance   {NARROWEST_CLEARANCE_M*100:5.1f} cm, half = {NARROWEST_CLEARANCE_M*50:.1f} cm")
print()
print("GATE: PASS" if d_blind <= NARROWEST_CLEARANCE_M / 2 else
      "GATE: CAPPED AT N = 1.00  -- slow down or speed the loop up")

json.dump({"f_median_hz": f_med, "f_worst5_hz": f_p05,
           "v_est_ms": v_est, "blind_distance_m": d_blind},
          open(DATA / "logs" / "latency.json", "w"), indent=2)

### Failsafe demonstration (gate)

Cover the camera. The car must stop within `STALE_S`. Record it — the marshal
will ask you to show this before the scored run.

In [ ]:
log.clear()
print("cover the lens when the loop starts...")
_ = drive_loop(duration_s=10.0)
stale = sum(e.get("stale", 0) for e in log)
print(f"{stale} stale-frame stops recorded out of {len(log)} iterations")
print("GATE: PASS" if stale > 0 else "GATE: FAIL — the failsafe never fired")

## 5 · The scored run

Three consecutive laps. A marshal times each lap and counts wall contacts. A
manual intervention voids the run.

Enter the timings below; the cell computes your Navigation Index and writes the
submission file the leaderboard reads.

In [ ]:
# ── filled in by the marshal ─────────────────────────────────────
T0_BASELINE_S = 24.8          # your car's stock-policy baseline, from the pre-session run
LAP_TIMES_S   = [21.4, 20.9, 21.7]
CONTACTS      = 1             # wall contacts across the three laps
INTERVENTIONS = 0             # any value > 0 voids the run
# ─────────────────────────────────────────────────────────────────

import statistics

assert len(LAP_TIMES_S) == 3, "three consecutive laps, not one"
assert INTERVENTIONS == 0, "run voided by manual intervention"

T_med   = statistics.median(LAP_TIMES_S)
T_adj   = T_med + 2.0 * CONTACTS
N_raw   = T0_BASELINE_S / T_adj
capped  = d_blind > NARROWEST_CLEARANCE_M / 2
N       = min(N_raw, 1.00) if capped else N_raw

submission = {
    "team": TEAM, "car": CAR,
    "baseline_T0_s": T0_BASELINE_S,
    "lap_times_s": LAP_TIMES_S,
    "T_median_s": T_med, "contacts": CONTACTS, "T_adjusted_s": T_adj,
    "loop_rate_hz": f_med, "blind_distance_m": d_blind,
    "latency_capped": capped,
    "held_out_mse": best,
    "navigation_index": N,
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
}
path = DATA / f"SUBMISSION_Ex11.1_{RUN_ID}.json"
path.write_text(json.dumps(submission, indent=2))

print(f"median lap        {T_med:6.2f} s")
print(f"contact penalty  +{2.0*CONTACTS:6.2f} s")
print(f"adjusted         {T_adj:6.2f} s   against baseline {T0_BASELINE_S:.2f} s")
print(f"NAVIGATION INDEX  {N:6.3f}" + ("   (CAPPED — blind distance)" if capped else ""))
print(f"\nwrote {path}")

## 6 · Hand in

Copy off the car:

- `SUBMISSION_Ex11.1_<team>_car<NN>.json`
- `logs/latency.json`, `logs/training.json`
- `model_best.pth`

And write two paragraphs: what limited you — data, latency, or grip — and how
you know. "It felt fast" is not an answer; `d = v/f` is.

**This becomes your Ex_11.2 baseline.** The policy you hand in here is the one
you will make efficient next week.